# Crawling + Ekstraksi Putusan Pidana Narkotika — PN Bangil (2023–2025)

## 0. Persiapan

In [1]:
%pip install -q pypdf playwright pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, csv, glob, time, random, shutil, zipfile
import pandas as pd
from urllib.parse import urljoin
from pypdf import PdfReader

# ---------------- SETELAN ----------------
PDF_DIR = "pdf"
TXT_DIR = "txt"
OUT_CSV = "hasil_ekstraksi.csv"
LIST_CSV = "daftar_unduhan.csv"
DEBUG_DIR = "debug"
PROFILE = os.path.abspath("chrome_profile")   # profil Chrome (verifikasi Cloudflare tersimpan di sini)
LEMBAGA_DEFAULT = "PN Bangil"

HOST = "https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-bangil/"
CATEGORY = HOST + "kategori/narkotika-dan-psikotropika-1.html"
BASE = HOST + "kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/{year}.html"
QUOTA = {2023: 20, 2024: 20, 2025: 20}    # total 60
SKIP_INKRACHT = True      # True = lewati putusan berstatus "tetap" (inkracht). False = ambil semua
LOCAL_DIR = "putusan_cadangan"   # folder lokal berisi PDF / ZIP (cadangan)

TARGET_TOTAL = sum(QUOTA.values())
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)
print("Folder kerja:", os.getcwd())
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

Folder kerja: d:\Semester 7\Informasi


## Fungsi nomor putusan, nama file, ekstraksi (model regex)

In [3]:
NO_RE = re.compile(r"(\d{1,5})\s*/\s*(P[A-Za-z]{1,3}\.?\s*[A-Za-z\-]*)\s*/\s*(\d{4})\s*/\s*(PN\.?\s*[A-Za-z]+)", re.I)


def fmt_no(m):
    a, b, c, d = m.groups()
    b = re.sub(r"\s+", "", b).title()
    parts = re.sub(r"\s+", " ", d).strip().split(" ")
    d = parts[0].upper() + " " + " ".join(x.capitalize() for x in parts[1:])
    return a + "/" + b + "/" + c + "/" + d.strip()


def make_name(no, ts):
    """'363/Pid.Sus/2024/PN Bil' -> putusan_363_pid.sus_2024_pn_bil_<ts>.pdf"""
    m = NO_RE.search(no or "")
    if not m:
        return None
    a, b, c, d = m.groups()
    jenis = re.sub(r"\s+", "", b).lower()
    pn = re.sub(r"[^a-z]+", "_", d.lower()).strip("_")
    return f"putusan_{a}_{jenis}_{c}_{pn}_{ts}.pdf"


def unique_name(name):
    base, ext = os.path.splitext(name)
    out, i = name, 2
    while os.path.exists(os.path.join(PDF_DIR, out)):
        out = f"{base}_{i}{ext}"
        i += 1
    return out


LEMBAGA_RE = re.compile(
    r"Pengadilan\s+Negeri\s+([A-Za-z]+(?:\s+[A-Za-z]+)??)\s+(?:yang|telah|dalam|memeriksa|mengadili|di)\b", re.I)
MENGADILI_RE = re.compile(r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?")
BB_RE = re.compile(
    r"barang\s+bukti\s*(?:berupa\s*:?|:)\s*(.+?)"
    r"(?=\s(?:Dirampas|Dimusnahkan|Dikembalikan|Membebankan|Membebani|Menetapkan|Memerintahkan|Demikian)\b|$)",
    re.I | re.S)
NOISE = [
    re.compile(r"Disclaimer.{0,1500}?ext\.?\s*318\)?", re.I | re.S),
    re.compile(r"Direktori\s+Putusan\s+Mahkamah\s+Agung\s+Republik\s+Indonesia", re.I),
    re.compile(r"putusan\d?\.mahkamahagung\.go\.id", re.I),
    re.compile(r"Putusan\s+Nomor\s+\d{1,5}\s*/\s*P[A-Za-z]{1,3}\.?[\w\-]*\s*/\s*\d{4}\s*/\s*PN\.?\s*[A-Za-z]+", re.I),
    re.compile(r"(?:Halaman|Hal\.?)\s*\d+\s+dari\s+\d+\s*(?:halaman|hal\.?)?", re.I),
]


def clean(t):
    for rx in NOISE:
        t = rx.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()


def get_amar(t):
    ms = list(MENGADILI_RE.finditer(t))
    if not ms:
        return ""
    rest = t[ms[-1].end():].strip()
    m = (re.search(r"\bDemikian(?:lah)?\s+(?:diputus|putusan|diputuskan|dijatuhkan|di putus)", rest)
         or re.search(r"\bDemikian(?:lah)?\b", rest))
    return (rest[:m.start()] if m else rest[:5000]).strip()


def get_bb(t, amar):
    m = BB_RE.search(amar) if amar else None
    if not m:
        allm = list(BB_RE.finditer(t))
        m = allm[-1] if allm else None
    return m.group(1).strip(" ;:-")[:2500] if m else ""


def extract_fields(raw, no_web=""):
    norm = re.sub(r"\s+", " ", raw)
    head = norm[:5000]
    m = NO_RE.search(head)
    no = fmt_no(m) if m else no_web
    m = LEMBAGA_RE.search(head)
    lembaga = ("PN " + m.group(1).title()) if m else LEMBAGA_DEFAULT
    t = clean(norm)
    amar = get_amar(t)
    return no, lembaga, get_bb(t, amar), amar, len(norm)


def read_pdf_text(fp):
    try:
        return "\n".join((pg.extract_text() or "") for pg in PdfReader(fp).pages)
    except Exception as e:
        print("gagal baca", os.path.basename(fp), "|", e)
        return ""

## Crawling dengan browser asli (Playwright)

membuka **Chrome asli** di laptop (bukan `requests`), melewati Cloudflare.

- Verifikasi tersimpan di folder `chrome_profile/`, jadi run berikutnya biasanya tidak diminta lagi.

In [4]:
# unduh Chromium bawaan Playwright, dipakai bila Google Chrome tidak terpasang
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-500:] or "selesai")


selesai


In [5]:
import asyncio, threading, sys, json
from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout

JS = "els => els.map(e => e.href)"
SEL = "a[href*='/direktori/putusan/']"
PDF_SEL = "a[href*='download_file'][href*='/pdf/'], a[href$='.pdf'], a[href*='/pdf/']"
META_JS = r"""() => {
  const out = {};
  document.querySelectorAll('table tr').forEach(tr => {
    const c = tr.querySelectorAll('th,td');
    if (c.length >= 2) {
      const k = c[0].innerText.trim().replace(/\s+/g, ' ');
      const v = Array.from(c).slice(1).map(x => x.innerText.trim().replace(/\s+/g, ' ')).join(' ');
      if (k && !(k in out)) out[k] = v;
    }
  });
  document.querySelectorAll('dt').forEach(dt => {
    const dd = dt.nextElementSibling;
    const k = dt.innerText.trim();
    if (dd && dd.tagName === 'DD' && k && !(k in out)) out[k] = dd.innerText.trim().replace(/\s+/g, ' ');
  });
  return out;
}"""


def is_challenge(pg):
    try:
        t = (pg.title() + " " + pg.inner_text("body")[:300]).lower()
    except Exception:
        return True
    return any(k in t for k in ("just a moment", "verify you are human", "verifying you are human",
                                "attention required", "checking your browser",
                                "performing security verification"))


def wait_clear(pg, maxwait=300):
    if not is_challenge(pg):
        return True
    print("  Verifikasi Cloudflare. Kalau ada kotak 'Verify', klik di browser. Saya tunggu otomatis...")
    t0, ok = time.time(), 0
    while time.time() - t0 < maxwait:
        time.sleep(2)
        ok = 0 if is_challenge(pg) else ok + 1
        if ok >= 2:
            print("  verifikasi lolos")
            return True
    print("  verifikasi tidak selesai dalam", maxwait, "detik")
    return False


def go(pg, url):
    try:
        pg.goto(url, wait_until="load", timeout=90000)
    except PWTimeout:
        print("  (load lambat, lanjut):", url)
    except Exception as e:
        msg = str(e)
        if "interrupted by another navigation" in msg:
            time.sleep(4)
        else:
            print("goto gagal:", url, "|", msg.splitlines()[0])
            return False
    if not wait_clear(pg):
        return False
    try:
        pg.wait_for_load_state("load", timeout=30000)
    except Exception:
        pass
    return True


def dump(pg, tag):
    try:
        body = re.sub(r"\s+", " ", pg.inner_text("body"))[:200]
        print("  [debug] title:", pg.title(), "| isi:", body)
        pg.screenshot(path=f"{DEBUG_DIR}/{tag}.png")
        open(f"{DEBUG_DIR}/{tag}.html", "w", encoding="utf-8").write(pg.content())
    except Exception as e:
        print("  dump gagal:", e)


def page_url(base, n):
    return base if n == 1 else base[:-5] + f"/page/{n}.html"


def list_links(pg, url, tag):
    if not go(pg, url):
        return []
    try:
        pg.wait_for_selector(SEL, timeout=15000, state="attached")
    except PWTimeout:
        pass
    time.sleep(random.uniform(2, 4))
    links = [l for l in dict.fromkeys(pg.eval_on_selector_all(SEL, JS)) if l.endswith(".html")]
    if not links:
        dump(pg, tag)
    return links


def last_page(pg):
    nums = []
    for a in pg.query_selector_all("a[href*='/page/']"):
        m = re.search(r"/page/(\d+)\.html", a.get_attribute("href") or "")
        if m:
            nums.append(int(m.group(1)))
    return max(nums) if nums else 1


def discover(pg):
    found = {}
    for a in pg.query_selector_all("a[href*='/tahun/']"):
        href = a.get_attribute("href") or ""
        m = re.search(r"/tahunjenis/putus/tahun/(\d{4})\.html", href)
        if not m:
            continue
        url = urljoin(CATEGORY, href)
        if "pn-bangil" not in url or "narkotika" not in url:
            continue
        y = int(m.group(1))
        if y not in QUOTA:      # abaikan tahun di luar target (mis. 2022, 2026)
            continue
        if url not in found.setdefault(y, []):
            found[y].append(url)
            print(f"  sidebar: {y} teks='{a.inner_text().strip()}'")
    return found


def candidates(year, found):
    c = list(found.get(year, []))
    u = BASE.format(year=year)
    if u not in c:
        c.append(u)
    return c


def fetch_pdf(pg, ctx, a, d, path):
    href = urljoin(d, a.get_attribute("href") or "")
    try:
        with pg.expect_download(timeout=60000) as dl:
            try:
                a.click(timeout=5000)
            except Exception:
                a.evaluate("e => e.click()")
        dl.value.save_as(path)
        return
    except Exception as e:
        print("  klik gagal, coba request langsung:", str(e).splitlines()[0])
    r = ctx.request.get(href, timeout=90000)
    body = r.body()
    if r.ok and body[:4] == b"%PDF":
        with open(path, "wb") as f:
            f.write(body)
    else:
        raise RuntimeError(f"bukan PDF (HTTP {r.status})")


def open_browser(p):
    kw = dict(user_data_dir=PROFILE, headless=False, accept_downloads=True,
              viewport=None, args=["--disable-blink-features=AutomationControlled"],
              ignore_default_args=["--enable-automation"])
    try:
        return p.chromium.launch_persistent_context(channel="chrome", **kw)
    except Exception as e:
        print("Chrome asli tidak ada, pakai Chromium bawaan:", str(e).splitlines()[0])
        return p.chromium.launch_persistent_context(**kw)


def get_meta(pg):
    try:
        return pg.evaluate(META_JS) or {}
    except Exception:
        return {}


def get_status(meta, body):
    for k, v in meta.items():
        if "status" in k.lower() and v:
            return v
    m = re.search(r"(Berkekuatan Hukum Tetap|Inkracht[^\n]{0,40}|Belum[^\n]{0,30}Hukum Tetap)", body, re.I)
    if m:
        return m.group(1).strip()
    for k, v in meta.items():
        if "tingkat proses" in k.lower() and v:
            return v + " (tingkat proses)"
    return "?"


def download_all():
    rows, got, seen, meta_shown = [], 0, set(), False

    def save_csv():
        with open(LIST_CSV, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(["No", "Tahun", "File", "Status", "URL", "NoWeb"])
            w.writerows(rows)

    with sync_playwright() as p:
        ctx = open_browser(p)
        pg = ctx.pages[0] if ctx.pages else ctx.new_page()
        go(pg, CATEGORY)
        time.sleep(3)
        print("Baca filter tahun di sidebar:")
        found = discover(pg)
        print("tahun di sidebar:", sorted(found) or "(tidak ketemu, pakai URL tebakan)")

        for year, quota in QUOTA.items():
            print(f"\n===== TAHUN {year} (target {quota}) =====")
            base, p1 = None, []
            for cand in candidates(year, found):
                print("coba:", cand)
                p1 = list_links(pg, page_url(cand, 1), f"list_{year}")
                if p1:
                    base = cand
                    break
            if not base:
                print(f"TAHUN {year}: kosong. Lihat folder debug. Lanjut tahun lain.")
                continue
            total_pages = last_page(pg)
            pages = {1: p1}
            if total_pages >= 2:
                pages[2] = list_links(pg, page_url(base, 2), f"list_{year}_p2")
            sidebar = set(p1) & set(pages.get(2, []))
            print("halaman:", total_pages, "| sidebar dibuang:", len(sidebar))

            year_got, n = 0, 1
            skipped_final = no_pdf = failed = other = 0
            pending = [l for l in p1 if l not in sidebar]
            while year_got < quota:
                if not pending:
                    n += 1
                    if n > total_pages:
                        print("halaman habis"); break
                    if n not in pages:
                        pages[n] = list_links(pg, page_url(base, n), f"list_{year}_p{n}")
                    pending = [l for l in pages[n] if l not in sidebar]
                    continue
                d = pending.pop(0)
                if d in seen:
                    continue
                seen.add(d)
                try:
                    if not go(pg, d):
                        failed += 1; continue
                    try:
                        pg.wait_for_selector(PDF_SEL, timeout=15000, state="attached")
                    except PWTimeout:
                        pass
                    body = pg.inner_text("body")
                    if "bangil" not in body.lower():
                        other += 1
                        print("bukan PN Bangil (sidebar?):", d); continue
                    meta = get_meta(pg)
                    if not meta_shown:
                        meta_shown = True
                        print("  [info] label tabel di halaman putusan:", list(meta))
                        with open(os.path.join(DEBUG_DIR, "meta_contoh.json"), "w", encoding="utf-8") as f:
                            json.dump({"url": d, "meta": meta}, f, ensure_ascii=False, indent=1)
                    status = get_status(meta, body)
                    mno = NO_RE.search(meta.get("Nomor", "")) or NO_RE.search(body)
                    no_web = fmt_no(mno) if mno else ""
                    if SKIP_INKRACHT and "tetap" in status.lower():
                        skipped_final += 1
                        print("lewati (inkracht):", d); continue
                    a = pg.query_selector(PDF_SEL)
                    if not a:
                        no_pdf += 1
                        print("tak ada link PDF:", d)
                        continue
                    ts = time.strftime("%Y%m%d%H%M%S")
                    fn = unique_name(make_name(no_web, ts) or f"putusan_{got + 1:02d}_{year}_{ts}.pdf")
                    fetch_pdf(pg, ctx, a, d, os.path.join(PDF_DIR, fn))
                    got += 1; year_got += 1
                    rows.append([got, year, fn, status, d, no_web]); save_csv()
                    print(got, fn, "|", no_web, "| status:", status)
                except Exception as e:
                    failed += 1
                    print("gagal:", d, e)
                time.sleep(random.uniform(3, 5))

            print(f"--- {year}: unduh {year_got}/{quota} | inkracht dilewati {skipped_final} "
                  f"| bukan Bangil {other} | tanpa PDF {no_pdf} | gagal {failed}")
        ctx.close()
    print("\nunduh selesai:", got, "putusan")




def jalankan_crawler():
    """Playwright sync API tidak boleh jalan di event loop Jupyter, jadi dijalankan di thread terpisah."""
    err = []

    def target():
        try:
            if sys.platform == "win32":
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
            download_all()
        except BaseException as e:
            err.append(e)

    t = threading.Thread(target=target)
    t.start()
    t.join()
    if err:
        print("\n[!] Crawler berhenti karena kesalahan:", type(err[0]).__name__, str(err[0])[:300])


jalankan_crawler()
print("\nTotal PDF di folder pdf/:", len(glob.glob(os.path.join(PDF_DIR, "*.pdf"))))


Baca filter tahun di sidebar:
  sidebar: 2025 teks='2025 263'
  sidebar: 2024 teks='2024 205'
  sidebar: 2023 teks='2023 199'
tahun di sidebar: [2023, 2024, 2025]

===== TAHUN 2023 (target 20) =====
coba: https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-bangil/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/2023.html
halaman: 10 | sidebar dibuang: 23
  [info] label tabel di halaman putusan: ['Nomor', 'Tingkat Proses', 'Klasifikasi', 'Kata Kunci', 'Tahun', 'Tanggal Register', 'Lembaga Peradilan', 'Jenis Lembaga Peradilan', 'Hakim Ketua', 'Hakim Anggota', 'Panitera', 'Amar', 'Amar Lainnya', 'Catatan Amar', 'Tanggal Musyawarah', 'Tanggal Dibacakan', 'Kaidah', 'Abstrak']
1 putusan_423_pid.sus_2023_pn_bil_20261104210156.pdf | 423/Pid.Sus/2023/PN Bil | status: Pertama (tingkat proses)
2 putusan_428_pid.sus_2023_pn_bil_20261104210201.pdf | 428/Pid.Sus/2023/PN Bil | status: Pertama (tingkat proses)
3 putusan_359_pid.sus_2023_pn_bil_20261104210205.pdf | 359/Pid.Su

## Ganti nama file ke format unduhan website MA

`putusan_<nomor>_<jenis>_<tahun>_<pn>_<yyyymmddHHMMSS>.pdf`
Contoh: `putusan_363_pid.sus_2024_pn_bil_20261104153012.pdf`. Nomor dibaca dari isi PDF. File yang sudah berformat itu dilewati.

In [6]:
SUDAH_BARU = re.compile(r"_\d{14}(?:_\d+)?\.pdf$")


def ganti_nama():
    n = 0
    for fp in sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf"))):
        fn = os.path.basename(fp)
        if SUDAH_BARU.search(fn):
            continue
        try:
            raw = "\n".join((pg.extract_text() or "") for pg in PdfReader(fp).pages[:2])
        except Exception:
            raw = ""
        m = NO_RE.search(re.sub(r"\s+", " ", raw)[:5000])
        no = fmt_no(m) if m else ""
        ts = time.strftime("%Y%m%d%H%M%S", time.localtime(os.path.getmtime(fp)))
        stem = re.sub(r"^putusan_", "", os.path.splitext(fn)[0])
        baru = unique_name(make_name(no, ts) or f"putusan_{stem}_{ts}.pdf")
        os.rename(fp, os.path.join(PDF_DIR, baru))
        print(fn, "->", baru)
        n += 1
    print("diganti:", n, "file")


ganti_nama()

diganti: 0 file


## Buat file TXT dari tiap PDF

Contoh: `pdf/putusan_363_pid.sus_2024_pn_bil_20261104153012.pdf` → `txt/putusan_363_pid.sus_2024_pn_bil_20261104153012.txt`


In [7]:
def buat_txt():
    n = kosong = 0
    for fp in sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf"))):
        out = os.path.join(TXT_DIR, os.path.splitext(os.path.basename(fp))[0] + ".txt")
        teks = read_pdf_text(fp).strip()
        with open(out, "w", encoding="utf-8") as f:
            f.write(teks)
        n += 1
        if len(teks) < 300:
            kosong += 1
            print("teks kosong/sedikit (PDF scan?):", os.path.basename(fp))
    print(f"TXT dibuat: {n} file di folder '{TXT_DIR}/' | teks kosong: {kosong}")


buat_txt()


TXT dibuat: 60 file di folder 'txt/' | teks kosong: 0


## Ekstraksi ke CSV

Aturan model (regex):
- **No Putusan** — pola `angka/Pid.Sus/tahun/PN Xxx` di awal dokumen (cadangan: dari nama file).
- **Lembaga Peradilan** — kalimat "Pengadilan Negeri ... yang memeriksa" → `PN <nama>`.
- **Amar Putusan** — teks setelah kata `MENGADILI` sampai kata `Demikian...`.
- **Barang Bukti** — teks setelah "barang bukti berupa" di dalam amar, sampai `Dirampas/Dimusnahkan/Dikembalikan/Membebankan`.
- Disclaimer MA, header, dan nomor halaman dibuang sebelum diproses.

In [8]:
def no_dari_nama(fn):
    m = re.match(r"putusan_(\d+)_([a-z.\-]+)_(\d{4})_([a-z_]+?)_\d{14}", fn)
    if not m:
        return ""
    a, b, c, d = m.groups()
    return a + "/" + b.title() + "/" + c + "/" + " ".join(
        [d.split("_")[0].upper()] + [x.capitalize() for x in d.split("_")[1:]])


def urut(fp):
    fn = os.path.basename(fp)
    m = re.match(r"putusan_(\d+)_[^_]+_(\d{4})_", fn)
    return (int(m.group(2)), int(m.group(1)), fn) if m else (9999, 0, fn)


def ekstraksi():
    rows, cek = [], []
    for i, fp in enumerate(sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")), key=urut), 1):
        fn = os.path.basename(fp)
        no, lembaga, bb, amar, n = extract_fields(read_pdf_text(fp), no_dari_nama(fn))
        rows.append([i, no, lembaga, bb, amar])
        cek.append({"File": fn, "teks_ada": n >= 300, "no": bool(no),
                    "barang_bukti": bool(bb), "amar": bool(amar)})
    df = pd.DataFrame(rows, columns=["No", "No Putusan", "Lembaga Peradilan", "Barang Bukti", "Amar Putusan"])
    return df, pd.DataFrame(cek)


df, cek = ekstraksi()
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(len(df), "putusan diekstraksi ->", OUT_CSV)
if len(cek):
    print("\nKolom terisi (True = berhasil):")
    print(cek[["teks_ada", "no", "barang_bukti", "amar"]].sum().to_string())
df.head(10)

60 putusan diekstraksi -> hasil_ekstraksi.csv

Kolom terisi (True = berhasil):
teks_ada        60
no              60
barang_bukti    59
amar            59


,No,No Putusan,Lembaga Peradilan,Barang Bukti,Amar Putusan
0,1,141/Pid.Sus/2023/PN Bil,PN Bangil," 1 (satu) kantong plastik kecil berisi Narkotika Golongan I Jenis Sabu dengan berat kotor 1,57 (satu koma lima puluh tujuh) gram;  1 (satu) buah...","1. Menyatakan Terdakwa AGUS ROBIYANTO Bin MUJIANTO tersebut diatas, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana 20 Mahkama..."
1,2,336/Pid.Sus/2023/PN Bil,PN Bangil,"4 (empat) poket narkotika jenis sabu dengan berat bruto 9,25 (sembilan koma dua lima) gram berikut plastik pembungkusnya - 1 (satu) poket Narkotik...",1. Menyatakan Terdakwa I Moh. Nur Syamsudin Bin Timat dan Terdakwa II Haris Iman Nursalam Bin Karsono telah terbukti secara sah dan meyakinkan ber...
2,3,357/Pid.Sus/2023/PN Bil,PN Bangil,"– 1 (satu) kantong plastic klip berisikan narkotika gol 1 jenis sabu dengan berat kotor 5,18 gram; – 1 (satu) buah Hp merk oppo warna hitam dnegan...","1. Menyatakan terdakwa IHSAN EFENDI Bin MUHAMMAD, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “tanpa hak atau melawan huku..."
3,4,359/Pid.Sus/2023/PN Bil,PN Bangil,"a. 3 (tiga) kantong plastik kecil berisi Narkotika Gol I jenis Sabu dengan berat kotor masing-masing 0,34 (nol koma tiga empat) gram, 1,10 (satu k...","1. Menyatakan Terdakwa Sugianto Bin Saiful Alim tersebut diatas, terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana “tanpa hak at..."
4,5,362/Pid.Sus/2023/PN Bil,PN Bangil," 3 (tiga) kantong plastik kecil berisi Narkotika Golongan I (Shabu) dengan berat kotor 0,36 (Nol Koma Tiga Puluh Enam) Gram, berat kotor 2,46 (Du...",1. Menyatakan Terdakwa Mashuda Bin M. Munir telah terbukti secara sah dan menyakinkan bersalah melakukan tindak pidana “Tanpa Hak Dan Melawan Huku...
5,6,376/Pid.Sus/2023/PN Bil,PN Bangil,"2 (dua) poket plastic kecil berisi narkotika Gol I jenis shabu dengan berat kotor masing-masing 0,92 (nol koma Sembilan dua) gram, 0,72 (nol koma ...",1. Menyatakan terdakwa Mustakim Bin Jualim tersebut telah terbukti secara sah dan meyakinkan bersalah melakukan perbuatan tindak pidana “Tanpa hak...
6,7,377/Pid.Sus/2023/PN.BIL,PN Bangil,"1 (satu) kantong plastik kecil berisi narkotika golongan I jenis shabu dengan berat kotor 1,11 (satu koma satu satu) gram.",1. Menyatakan Terdakwa AKHMAD MUHAJIR ZAIN Alias WAWAN Bin BAKIRNO (Alm) telah terbukti secara sah dan meyakinkan bersalah melakukan tindak pidana...
7,8,380/Pid.Sus/2023/PN Bil,PN Bangil,"1 pipet kaca berisi sisa sabu dengan berat kotor 2,22 gram, kartu SIM IM3 dengan nomor 085771252376 dan XL dengan nomor 087843300432,",1. Menyatakan terdakwa Ahmad Harianto Bin Ponasit tersebut telah terbukti secara sah dan meyakinkan bersalah melakukan perbuatan tindak pidana “ T...
8,9,381/Pid.Sus/2023/PN Bil,PN Bangil,"1 (satu) handphone merk OPPO warna merah kartu sim IM3 dengan nomor 085856112232,",1. Menyatakan terdakwa Untung Slamet Bin Lumadi (alm) tersebut telah terbukti secara sah dan meyakinkan bersalah melakukan perbuatan tindak 19 Mah...
9,10,393/Pid.Sus/2023/PN Bil,PN Bangil," 16 (enam) belas bungkus plastik klip berisi sabu dengan berat kotor 2,81 (dua koma delapan satu) gram, 2 (satu) buah dompet kecil berwarna hitam...","1. Menyatakan terdakwa Moh. Fauzi Bin Arpai Alm tersebut telah terbukti secara sah dan meyakinkan bersalah melakukan perbuatan tindak pidana ""Tanp..."


### Cek baris bermasalah (kolom kosong / PDF hasil scan)


In [9]:
if len(cek):
    bermasalah = cek[~(cek.teks_ada & cek.no & cek.barang_bukti & cek.amar)]
    print("semua kolom terisi" if bermasalah.empty else bermasalah.to_string())
else:
    print("belum ada PDF. Isi folder putusan_cadangan/ (bagian 3) lalu ulangi bagian 3-5.")

                                                  File  teks_ada    no  barang_bukti   amar
15  putusan_430_pid.sus_2023_pn_bil_20261104210312.pdf      True  True         False  False


## Hasil


In [10]:
print("File hasil :", os.path.abspath(OUT_CSV))
print("Folder PDF  :", os.path.abspath(PDF_DIR), "->", len(glob.glob(os.path.join(PDF_DIR, "*.pdf"))), "file")
print("Folder TXT  :", os.path.abspath(TXT_DIR), "->", len(glob.glob(os.path.join(TXT_DIR, "*.txt"))), "file")


File hasil : d:\Semester 7\Informasi\hasil_ekstraksi.csv
Folder PDF  : d:\Semester 7\Informasi\pdf -> 60 file
Folder TXT  : d:\Semester 7\Informasi\txt -> 60 file


In [11]:
import os, glob, zipfile

ZIP_NAME = "putusan_Narkotika_PN-Bangil.zip"

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder, pola in ((PDF_DIR, "*.pdf"), (TXT_DIR, "*.txt")):
        for fp in sorted(glob.glob(os.path.join(folder, pola))):
            z.write(fp, arcname=os.path.join(folder, os.path.basename(fp)))

print("ZIP dibuat :", os.path.abspath(ZIP_NAME))
print("Ukuran     :", round(os.path.getsize(ZIP_NAME) / 1024 / 1024, 2), "MB")
with zipfile.ZipFile(ZIP_NAME) as z:
    print("Isi        :", len(z.namelist()), "file")

ZIP dibuat : d:\Semester 7\Informasi\putusan_Narkotika_PN-Bangil.zip
Ukuran     : 12.15 MB
Isi        : 120 file
